In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time
import json
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

try:
    from PIL import Image, ImageDraw, ImageFont
    _PIL_OK = True
except Exception:
    _PIL_OK = False


# ============================================================
# CONFIG
# ============================================================
RE_MAX = 30.0
DEVICE = torch.device("cpu")
DTYPE = torch.float64

X_MIN_ND = 0.0
X_MAX_ND = 22.0

FILTER_SHAPE: Optional[str] = None
FILTER_RE: Optional[float] = None

DEBUG_PRINTS: bool = False

CODE_DIR = Path(r"D:\Projects\Simulations\AI FInal\Code\Main Structure")
DATA_ROOT = Path(r"D:\Projects\Simulations\AI FInal\Ansys\Shapes")
OUT_DIR = Path(r"D:\Projects\Simulations\AI FInal\Code\Results\Evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = CODE_DIR / "checkpoints_stage5" / "stage5_pinn_cpu.pt"


# ============================================================
# COLORS
# ============================================================
FIG_BG = "#f0f8ff"
C_PRIMARY = "#ff033e"

plt.rcParams["figure.facecolor"] = FIG_BG
plt.rcParams["savefig.facecolor"] = FIG_BG


# ============================================================
# IMPORTS
# ============================================================
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from pinn_loader import load_all_cases
from stage5_model import FlowPINNHardBC
from audit_cases import audit_all


# ============================================================
# HELPERS
# ============================================================
def read_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def pick_first_numeric(d: Dict[str, Any], keys: List[str]) -> float:
    for k in keys:
        if k in d:
            try:
                v = d[k]
                if v is None:
                    continue
                if isinstance(v, str) and v.strip() == "":
                    continue
                fv = float(v)
                if np.isfinite(fv):
                    return fv
            except Exception:
                continue
    return float("nan")


def infer_walls_from_cases(cases) -> Tuple[float, float]:
    ys = []
    for c in cases:
        Xw = c.bc["wall"]["X"]
        if Xw is not None and len(Xw) > 0:
            ys.append(np.asarray(Xw)[:, 1].astype(np.float64))

    if len(ys) == 0:
        raise RuntimeError("No bc_wall points found; cannot infer y_bot/y_top.")

    y_all = np.concatenate(ys, axis=0)
    y_bot = float(np.min(y_all))
    y_top = float(np.max(y_all))

    if not (y_top > y_bot):
        raise RuntimeError(f"Bad inferred walls: y_bot={y_bot}, y_top={y_top}")

    return y_bot, y_top


def torch_load_trusted(path: Path):
    return torch.load(path, map_location="cpu", weights_only=False)


def load_model_from_ckpt(
    ckpt_path: Path,
    y_bot: float,
    y_top: float,
    x_min: float,
    x_max: float,
    dtype=torch.float64,
):
    ckpt = torch_load_trusted(ckpt_path)
    if not isinstance(ckpt, dict) or "model" not in ckpt:
        raise RuntimeError(f"Checkpoint format unexpected: {ckpt_path}")

    cfg = ckpt.get("cfg", {}) if isinstance(ckpt, dict) else {}

    width = int(cfg.get("width", 128))
    n_blocks = int(cfg.get("n_blocks", 6))
    activation = str(cfg.get("activation", "tanh"))
    beta_body = float(cfg.get("beta_body", 5.0))
    beta_wall = float(cfg.get("beta_wall", 5.0))
    softplus_k = float(cfg.get("softplus_k", 20.0))
    out_gain = float(cfg.get("out_gain", 0.1))

    model = FlowPINNHardBC(
        y_bot=y_bot,
        y_top=y_top,
        x_min=float(x_min),
        x_max=float(x_max),
        width=width,
        n_blocks=n_blocks,
        activation=activation,
        beta_body=beta_body,
        beta_wall=beta_wall,
        softplus_k=softplus_k,
        out_gain=out_gain,
    )
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model.to(dtype=dtype, device=DEVICE)


def stamp_text_on_png(png_path: Path, text: str):
    if not _PIL_OK:
        return

    img = Image.open(str(png_path)).convert("RGB")
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("arial.ttf", 18)
    except Exception:
        font = ImageFont.load_default()

    pad = 10
    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    _, H = img.size

    x = pad
    y = H - th - pad

    draw.rectangle([x - 6, y - 4, x + tw + 6, y + th + 4], fill=(0, 0, 0))
    draw.text((x, y), text, fill=(255, 255, 255), font=font)
    img.save(str(png_path))


def save_table_png(
    df: pd.DataFrame,
    title: str,
    out_path: Path,
    font_size: int = 11,
    row_scale: float = 1.6,
    col_scale: float = 1.2,
):
    fig_h = 0.60 * (len(df) + 1) + 1.8
    fig, ax = plt.subplots(figsize=(13.5, fig_h), dpi=230)
    fig.patch.set_facecolor(FIG_BG)
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        rowLabels=df.index.tolist(),
        colLabels=df.columns.tolist(),
        loc="center",
        cellLoc="center",
        rowLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(col_scale, row_scale)

    ncols = len(df.columns)
    for j in range(ncols):
        cell = table[(0, j)]
        cell.set_facecolor(C_PRIMARY)
        cell.set_text_props(color="black", weight="bold")

    for i in range(1, len(df) + 1):
        if (i, -1) in table.get_celld():
            rl = table[(i, -1)]
            rl.set_text_props(weight="bold")
            rl.set_facecolor("#f2f2f2" if (i % 2 == 0) else "#ffffff")

        for j in range(ncols):
            c = table[(i, j)]
            c.set_facecolor("#f7f7f7" if (i % 2 == 0) else "#ffffff")

    ax.set_title(title, fontsize=14, pad=14)
    fig.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def compute_metrics(Y_pred: np.ndarray, Y_true: np.ndarray):
    err = Y_pred - Y_true
    rmse = np.sqrt(np.mean(err**2, axis=0))
    relL2 = np.linalg.norm(err, axis=0) / (np.linalg.norm(Y_true, axis=0) + 1e-12)

    Vp = np.sqrt(Y_pred[:, 0] ** 2 + Y_pred[:, 1] ** 2)
    Vt = np.sqrt(Y_true[:, 0] ** 2 + Y_true[:, 1] ** 2)
    Verr = Vp - Vt
    rmse_V = float(np.sqrt(np.mean(Verr**2)))
    relL2_V = float(np.linalg.norm(Verr) / (np.linalg.norm(Vt) + 1e-12))
    return rmse, relL2, rmse_V, relL2_V


def build_grid_fields(x, y, phi, z, nx=900, ny=220):
    tri = mtri.Triangulation(x, y)
    tris = tri.triangles
    tri_mask = np.any(phi[tris] <= 0.0, axis=1)
    tri.set_mask(tri_mask)

    interp_z = mtri.LinearTriInterpolator(tri, z)
    interp_phi = mtri.LinearTriInterpolator(tri, phi)

    xi = np.linspace(float(np.min(x)), float(np.max(x)), nx)
    yi = np.linspace(float(np.min(y)), float(np.max(y)), ny)
    XI, YI = np.meshgrid(xi, yi)

    ZI = interp_z(XI, YI)
    PHII = interp_phi(XI, YI)
    ZI = np.ma.array(ZI, mask=np.isnan(ZI) | (PHII <= 0.0))
    return xi, yi, ZI


def save_3x3_fields(
    x, y, phi,
    u_cfd, v_cfd, p_cfd,
    u_pinn, v_pinn, p_pinn,
    tag: str,
    out_path: Path,
    body_xy=None,
    infer_s: Optional[float] = None,
):
    ue = np.abs(u_pinn - u_cfd)
    ve = np.abs(v_pinn - v_cfd)
    pe = np.abs(p_pinn - p_cfd)

    xi, yi, Uc = build_grid_fields(x, y, phi, u_cfd)
    _, _, Up = build_grid_fields(x, y, phi, u_pinn)
    _, _, Ue = build_grid_fields(x, y, phi, ue)

    _, _, Vc = build_grid_fields(x, y, phi, v_cfd)
    _, _, Vp = build_grid_fields(x, y, phi, v_pinn)
    _, _, Ve = build_grid_fields(x, y, phi, ve)

    _, _, Pc = build_grid_fields(x, y, phi, p_cfd)
    _, _, Pp = build_grid_fields(x, y, phi, p_pinn)
    _, _, Pe = build_grid_fields(x, y, phi, pe)

    umin = float(np.nanmin([np.nanmin(u_cfd), np.nanmin(u_pinn)]))
    umax = float(np.nanmax([np.nanmax(u_cfd), np.nanmax(u_pinn)]))
    vmin = float(np.nanmin([np.nanmin(v_cfd), np.nanmin(v_pinn)]))
    vmax = float(np.nanmax([np.nanmax(v_cfd), np.nanmax(v_pinn)]))
    pmin = float(np.nanmin([np.nanmin(p_cfd), np.nanmin(p_pinn)]))
    pmax = float(np.nanmax([np.nanmax(p_cfd), np.nanmax(p_pinn)]))

    emax_u = float(np.nanmax(ue)) if np.isfinite(np.nanmax(ue)) else 1.0
    emax_v = float(np.nanmax(ve)) if np.isfinite(np.nanmax(ve)) else 1.0
    emax_p = float(np.nanmax(pe)) if np.isfinite(np.nanmax(pe)) else 1.0

    fig, axes = plt.subplots(3, 3, figsize=(18, 11), dpi=230)
    fig.patch.set_facecolor(FIG_BG)

    for ax in axes.ravel():
        ax.set_aspect("equal", adjustable="box")
        ax.set_axis_off()

    axes[0, 0].imshow(Uc, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=umin, vmax=umax)
    axes[0, 0].set_title(f"CFD u_nd | {tag}", fontsize=11)
    axes[0, 1].imshow(Up, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=umin, vmax=umax)
    axes[0, 1].set_title(f"PINN u_nd | {tag}", fontsize=11)
    axes[0, 2].imshow(Ue, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=0.0, vmax=emax_u)
    axes[0, 2].set_title(f"|err u| | {tag}", fontsize=11)

    axes[1, 0].imshow(Vc, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=vmin, vmax=vmax)
    axes[1, 0].set_title(f"CFD v_nd | {tag}", fontsize=11)
    axes[1, 1].imshow(Vp, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=vmin, vmax=vmax)
    axes[1, 1].set_title(f"PINN v_nd | {tag}", fontsize=11)
    axes[1, 2].imshow(Ve, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=0.0, vmax=emax_v)
    axes[1, 2].set_title(f"|err v| | {tag}", fontsize=11)

    axes[2, 0].imshow(Pc, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=pmin, vmax=pmax)
    axes[2, 0].set_title(f"CFD p_nd | {tag}", fontsize=11)
    axes[2, 1].imshow(Pp, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=pmin, vmax=pmax)
    axes[2, 1].set_title(f"PINN p_nd | {tag}", fontsize=11)
    axes[2, 2].imshow(Pe, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=0.0, vmax=emax_p)
    axes[2, 2].set_title(f"|err p| | {tag}", fontsize=11)

    if body_xy is not None and body_xy[0].size > 0:
        for ax in axes.ravel():
            ax.scatter(body_xy[0], body_xy[1], s=2, c=C_PRIMARY)

    if infer_s is not None:
        fig.suptitle(f"{tag} | inference={infer_s:.3f}s", fontsize=12, y=0.995)

    fig.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def save_vmag_1x3(
    x, y, phi,
    V_cfd, V_pinn,
    tag: str,
    out_path: Path,
    body_xy=None,
    infer_s: Optional[float] = None,
):
    Ve = np.abs(V_pinn - V_cfd)

    xi, yi, Vc = build_grid_fields(x, y, phi, V_cfd)
    _, _, Vp = build_grid_fields(x, y, phi, V_pinn)
    _, _, Ve_g = build_grid_fields(x, y, phi, Ve)

    vmin = float(np.nanmin([np.nanmin(V_cfd), np.nanmin(V_pinn)]))
    vmax = float(np.nanmax([np.nanmax(V_cfd), np.nanmax(V_pinn)]))
    emax = float(np.nanmax(Ve)) if np.isfinite(np.nanmax(Ve)) else 1.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=230)
    fig.patch.set_facecolor(FIG_BG)

    for ax in axes:
        ax.set_aspect("equal", adjustable="box")
        ax.set_axis_off()

    axes[0].imshow(Vc, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=vmin, vmax=vmax)
    axes[0].set_title(f"CFD |V|_nd | {tag}", fontsize=11)

    axes[1].imshow(Vp, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=vmin, vmax=vmax)
    axes[1].set_title(f"PINN |V|_nd | {tag}", fontsize=11)

    axes[2].imshow(Ve_g, origin="lower", extent=[xi.min(), xi.max(), yi.min(), yi.max()], vmin=0.0, vmax=emax)
    axes[2].set_title(f"|err |V|| | {tag}", fontsize=11)

    if body_xy is not None and body_xy[0].size > 0:
        for ax in axes:
            ax.scatter(body_xy[0], body_xy[1], s=2, c=C_PRIMARY)

    if infer_s is not None:
        fig.suptitle(f"{tag} | inference={infer_s:.3f}s", fontsize=12, y=0.995)

    fig.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def get_case_summary(case):
    nd_dir = Path(case.paths.nd_dir)
    cs_path = nd_dir / "case_summary.json"
    if not cs_path.exists():
        raise FileNotFoundError(f"Missing case_summary.json at: {cs_path}")
    return read_json(cs_path)


def order_boundary_by_angle(x, y):
    cx = float(np.mean(x))
    cy = float(np.mean(y))
    ang = np.arctan2(y - cy, x - cx)
    return np.argsort(ang)


def polygon_signed_area(x, y):
    x2 = np.r_[x, x[0]]
    y2 = np.r_[y, y[0]]
    return 0.5 * float(np.sum(x2[:-1] * y2[1:] - x2[1:] * y2[:-1]))


def geom_normals_and_ds(x, y):
    x_prev = np.roll(x, 1)
    y_prev = np.roll(y, 1)
    x_next = np.roll(x, -1)
    y_next = np.roll(y, -1)

    tx = x_next - x_prev
    ty = y_next - y_prev
    tlen = np.sqrt(tx * tx + ty * ty) + 1e-12
    tx /= tlen
    ty /= tlen

    nx = ty
    ny = -tx

    area = polygon_signed_area(x, y)
    if area < 0.0:
        nx = -nx
        ny = -ny

    nlen = np.sqrt(nx * nx + ny * ny) + 1e-12
    nx /= nlen
    ny /= nlen

    dx_prev = x - x_prev
    dy_prev = y - y_prev
    dx_next = x_next - x
    dy_next = y_next - y
    ds = 0.5 * (
        np.sqrt(dx_prev * dx_prev + dy_prev * dy_prev) +
        np.sqrt(dx_next * dx_next + dy_next * dy_next)
    )
    return nx, ny, ds


def interpolate_phi_at_points(case, xy_nd: np.ndarray) -> np.ndarray:
    X = case.field_X.astype(np.float64)
    x = X[:, 0]
    y = X[:, 1]
    phi = X[:, 3]

    tri = mtri.Triangulation(x, y)
    interp_phi = mtri.LinearTriInterpolator(tri, phi)

    phi_q = interp_phi(xy_nd[:, 0], xy_nd[:, 1])
    phi_q = np.asarray(phi_q, dtype=np.float64)

    for i in range(len(phi_q)):
        if not np.isfinite(phi_q[i]):
            dx = x - xy_nd[i, 0]
            dy = y - xy_nd[i, 1]
            j = int(np.argmin(dx * dx + dy * dy))
            phi_q[i] = float(phi[j])

    return phi_q


def predict_model_with_grads(model, X_np: np.ndarray):
    X = torch.tensor(X_np, dtype=DTYPE, device=DEVICE, requires_grad=True)
    Y = model(X)

    u = Y[:, 0:1]
    v = Y[:, 1:2]
    p = Y[:, 2:3]

    du = torch.autograd.grad(u, X, grad_outputs=torch.ones_like(u), create_graph=False, retain_graph=True)[0]
    dv = torch.autograd.grad(v, X, grad_outputs=torch.ones_like(v), create_graph=False, retain_graph=True)[0]

    u_x = du[:, 0:1]
    u_y = du[:, 1:2]
    v_x = dv[:, 0:1]
    v_y = dv[:, 1:2]

    return (
        u.detach().cpu().numpy().ravel(),
        v.detach().cpu().numpy().ravel(),
        p.detach().cpu().numpy().ravel(),
        u_x.detach().cpu().numpy().ravel(),
        u_y.detach().cpu().numpy().ravel(),
        v_x.detach().cpu().numpy().ravel(),
        v_y.detach().cpu().numpy().ravel(),
    )


def lsq_value_and_grad(x, y, z, xq, yq, k=24):
    xq = np.asarray(xq, dtype=np.float64)
    yq = np.asarray(yq, dtype=np.float64)

    val = np.empty(len(xq), dtype=np.float64)
    zx = np.empty(len(xq), dtype=np.float64)
    zy = np.empty(len(xq), dtype=np.float64)

    n = len(x)
    kk = min(int(k), n)

    for i in range(len(xq)):
        d2 = (x - xq[i]) ** 2 + (y - yq[i]) ** 2
        j = np.argpartition(d2, kk - 1)[:kk]

        dx = x[j] - xq[i]
        dy = y[j] - yq[i]

        A = np.column_stack([np.ones_like(dx), dx, dy])
        w = 1.0 / np.maximum(np.sqrt(d2[j]), 1e-10)

        Aw = A * w[:, None]
        bw = z[j] * w

        coef, *_ = np.linalg.lstsq(Aw, bw, rcond=None)
        val[i], zx[i], zy[i] = coef[0], coef[1], coef[2]

    return val, zx, zy


def cfd_cdcl_from_body_offset(
    case,
    Xb_np: np.ndarray,
    Re: float,
    y_bot: float,
    y_top: float,
    eps_list=(1e-3, 3e-3, 1e-2, 3e-2),
    phi_min=1e-6,
):
    xb = Xb_np[:, 0].astype(np.float64)
    yb = Xb_np[:, 1].astype(np.float64)

    idx = order_boundary_by_angle(xb, yb)
    xb = xb[idx]
    yb = yb[idx]

    nx, ny, ds = geom_normals_and_ds(xb, yb)

    eps_pick = None
    for eps in eps_list:
        x_off = xb + eps * nx
        y_off = np.clip(yb + eps * ny, y_bot + 1e-6, y_top - 1e-6)
        phi_off = interpolate_phi_at_points(case, np.column_stack([x_off, y_off]))
        if np.mean(phi_off > phi_min) > 0.95:
            eps_pick = float(eps)
            break

    if eps_pick is None:
        eps_pick = float(eps_list[-1])

    x_off = xb + eps_pick * nx
    y_off = np.clip(yb + eps_pick * ny, y_bot + 1e-6, y_top - 1e-6)

    X = case.field_X.astype(np.float64)
    Y = case.field_Y.astype(np.float64)

    m = X[:, 3] > phi_min
    xg = X[m, 0]
    yg = X[m, 1]
    ug = Y[m, 0]
    vg = Y[m, 1]
    pg = Y[m, 2]

    p, _, _ = lsq_value_and_grad(xg, yg, pg, x_off, y_off)
    _, u_x, u_y = lsq_value_and_grad(xg, yg, ug, x_off, y_off)
    _, v_x, v_y = lsq_value_and_grad(xg, yg, vg, x_off, y_off)

    invRe = 1.0 / float(max(Re, 1e-12))
    t_x = (-p * nx) + invRe * (2.0 * u_x * nx + (u_y + v_x) * ny)
    t_y = (-p * ny) + invRe * ((u_y + v_x) * nx + 2.0 * v_y * ny)

    Fx = float(np.sum(t_x * ds))
    Fy = float(np.sum(t_y * ds))

    Cd = 2.0 * Fx
    Cl = 2.0 * Fy
    return Cd, Cl


def pinn_cdcl_from_body_offset(
    model,
    case,
    Xb_np: np.ndarray,
    Re: float,
    y_bot: float,
    y_top: float,
    eps_list=(1e-3, 3e-3, 1e-2, 3e-2),
    phi_min=1e-6,
):
    xb = Xb_np[:, 0].astype(np.float64)
    yb = Xb_np[:, 1].astype(np.float64)

    idx = order_boundary_by_angle(xb, yb)
    xb = xb[idx]
    yb = yb[idx]

    nx, ny, ds = geom_normals_and_ds(xb, yb)

    eps_pick = None
    for eps in eps_list:
        x_off = xb + eps * nx
        y_off = np.clip(yb + eps * ny, y_bot + 1e-6, y_top - 1e-6)
        phi_off = interpolate_phi_at_points(case, np.column_stack([x_off, y_off]))
        if np.mean(phi_off > phi_min) > 0.95:
            eps_pick = float(eps)
            break

    if eps_pick is None:
        eps_pick = float(eps_list[-1])

    x_off = xb + eps_pick * nx
    y_off = np.clip(yb + eps_pick * ny, y_bot + 1e-6, y_top - 1e-6)
    phi_off = interpolate_phi_at_points(case, np.column_stack([x_off, y_off]))

    Re_norm = float(Re) / float(RE_MAX)

    X_eval = np.zeros((len(x_off), 4), dtype=np.float64)
    X_eval[:, 0] = x_off
    X_eval[:, 1] = y_off
    X_eval[:, 2] = Re_norm
    X_eval[:, 3] = phi_off

    _, _, p, u_x, u_y, v_x, v_y = predict_model_with_grads(model, X_eval)

    invRe = 1.0 / float(max(Re, 1e-12))
    t_x = (-p * nx) + invRe * (2.0 * u_x * nx + (u_y + v_x) * ny)
    t_y = (-p * ny) + invRe * ((u_y + v_x) * nx + 2.0 * v_y * ny)

    Fx = float(np.sum(t_x * ds))
    Fy = float(np.sum(t_y * ds))

    Cd = 2.0 * Fx
    Cl = 2.0 * Fy
    return Cd, Cl


def point2_to_tuple(value: Any) -> Optional[Tuple[float, float]]:
    if isinstance(value, (list, tuple)) and len(value) == 2:
        try:
            a = float(value[0])
            b = float(value[1])
            if np.isfinite(a) and np.isfinite(b):
                return a, b
        except Exception:
            return None

    if isinstance(value, dict):
        for kx, ky in [("x", "y"), ("x_nd", "y_nd"), ("x_m", "y_m")]:
            if kx in value and ky in value:
                try:
                    a = float(value[kx])
                    b = float(value[ky])
                    if np.isfinite(a) and np.isfinite(b):
                        return a, b
                except Exception:
                    return None
    return None


def get_reference_probe_points_nd(cs: Dict[str, Any]) -> Tuple[Optional[Tuple[float, float]], Optional[Tuple[float, float]]]:
    rp = cs.get("reference_points", {})
    if not isinstance(rp, dict):
        return None, None

    pf = point2_to_tuple(rp.get("deltap_front_nd"))
    pr = point2_to_tuple(rp.get("deltap_rear_nd"))
    return pf, pr


def deltap_from_probe_points_pinn(model, case, front_nd: Tuple[float, float], rear_nd: Tuple[float, float], Re_case: float):
    xy = np.array([front_nd, rear_nd], dtype=np.float64)
    phi_q = interpolate_phi_at_points(case, xy)

    Re_norm = float(Re_case) / float(RE_MAX)
    Xq = np.zeros((2, 4), dtype=np.float64)
    Xq[:, 0] = xy[:, 0]
    Xq[:, 1] = xy[:, 1]
    Xq[:, 2] = Re_norm
    Xq[:, 3] = phi_q

    with torch.no_grad():
        pq = model(torch.tensor(Xq, dtype=DTYPE, device=DEVICE)).cpu().numpy()[:, 2]

    return float(pq[0] - pq[1])


def deltap_from_probe_points_cfd_field(case, front_nd: Tuple[float, float], rear_nd: Tuple[float, float]):
    X = case.field_X.astype(np.float64)
    Y = case.field_Y.astype(np.float64)

    x = X[:, 0]
    y = X[:, 1]
    phi = X[:, 3]
    p = Y[:, 2]

    tri = mtri.Triangulation(x, y)
    tris = tri.triangles
    tri_mask = np.any(phi[tris] <= 0.0, axis=1)
    tri.set_mask(tri_mask)

    interp_p = mtri.LinearTriInterpolator(tri, p)

    pf = interp_p(front_nd[0], front_nd[1])
    pr = interp_p(rear_nd[0], rear_nd[1])

    pf = float(pf) if np.isfinite(pf) else np.nan
    pr = float(pr) if np.isfinite(pr) else np.nan

    if not np.isfinite(pf) or not np.isfinite(pr):
        xy = np.array([front_nd, rear_nd], dtype=np.float64)
        out = []
        for i in range(2):
            dx = x - xy[i, 0]
            dy = y - xy[i, 1]
            j = int(np.argmin(dx * dx + dy * dy))
            out.append(float(p[j]))
        pf, pr = out[0], out[1]

    return float(pf - pr)


def get_centerline_y_nd(cs: Dict[str, Any], D_m: float, y_bot: float, y_top: float) -> float:
    scales = cs.get("scales", {})
    if isinstance(scales, dict):
        if "center_y_m" in scales:
            try:
                return float(scales["center_y_m"]) / float(D_m)
            except Exception:
                pass
    return 0.5 * (float(y_bot) + float(y_top))


def get_xe_nd(cs: Dict[str, Any]) -> float:
    fv = cs.get("final_values", {})
    if isinstance(fv, dict):
        xe_nd = pick_first_numeric(fv, ["xe_nd"])
        if np.isfinite(xe_nd):
            return float(xe_nd)

    rp = cs.get("reference_points", {})
    if isinstance(rp, dict):
        xe_nd = pick_first_numeric(rp, ["xe_nd"])
        if np.isfinite(xe_nd):
            return float(xe_nd)

    return float("nan")


def La_from_centerline_field(x, y, phi, u_vals, y_center_nd: float, xe_nd: float, nx=1400):
    if not np.isfinite(xe_nd):
        return float("nan")

    tri = mtri.Triangulation(x, y)
    tris = tri.triangles
    tri_mask = np.any(phi[tris] <= 0.0, axis=1)
    tri.set_mask(tri_mask)

    interp_u = mtri.LinearTriInterpolator(tri, u_vals)

    xmin = max(float(np.min(x)), float(xe_nd))
    xmax = float(np.max(x))
    xs = np.linspace(xmin, xmax, int(nx))
    ys = np.full_like(xs, float(y_center_nd))

    us = np.array(interp_u(xs, ys), dtype=np.float64)
    ok = np.isfinite(us)
    xs = xs[ok]
    us = us[ok]

    if len(xs) < 20:
        return float("nan")

    neg = us < 0.0
    if not np.any(neg):
        return float("nan")

    i_end = np.where(neg)[0][-1]
    if i_end >= len(xs) - 1:
        return float("nan")

    if us[i_end + 1] == us[i_end]:
        return float("nan")

    x0 = xs[i_end] + (0.0 - us[i_end]) * (xs[i_end + 1] - xs[i_end]) / (us[i_end + 1] - us[i_end] + 1e-14)
    return float(x0 - float(xe_nd))


# ============================================================
# MAIN — AUDIT / LOAD / MODEL
# ============================================================
print("Using checkpoint:", CKPT_PATH, CKPT_PATH.exists())
print("OUT_DIR:", OUT_DIR)

report = DATA_ROOT / "_audit_report.csv"
audit_all(DATA_ROOT, report)
print("[AUDIT] report saved:", report)

cases = load_all_cases(DATA_ROOT, Re_max=RE_MAX, verbose=True)

if FILTER_SHAPE is not None:
    cases = [c for c in cases if str(c.paths.shape) == str(FILTER_SHAPE)]
if FILTER_RE is not None:
    cases = [c for c in cases if float(c.paths.Re) == float(FILTER_RE)]

if not cases:
    print("[EVAL] No valid PASS cases found (after filters). Exiting (OK).")
    raise SystemExit(0)

print("[EVAL] PASS cases:", [(c.paths.shape, c.paths.Re, c.paths.case_id) for c in cases])

y_bot, y_top = infer_walls_from_cases(cases)
print(f"[WALLS] y_bot={y_bot:.6f}, y_top={y_top:.6f} (nd)")
print(f"[XDOMAIN] x_min={X_MIN_ND:.6f}, x_max={X_MAX_ND:.6f} (nd)")

model = load_model_from_ckpt(CKPT_PATH, y_bot, y_top, X_MIN_ND, X_MAX_ND, dtype=DTYPE)
print("[MODEL] Loaded checkpoint; evaluation uses CPU float64.")


# ============================================================
# PART 1 — FIELD METRICS + FIGURES
# ============================================================
metrics_rows = []
t_eval0 = time.perf_counter()

for c in cases:
    tag = f"{c.paths.shape}_Re{c.paths.Re}"

    X = c.field_X.astype(np.float64)
    Yt = c.field_Y.astype(np.float64)

    Xt = torch.tensor(X, dtype=DTYPE, device=DEVICE)

    t0 = time.perf_counter()
    with torch.no_grad():
        Yp = model(Xt).cpu().numpy()
    infer_s = float(time.perf_counter() - t0)

    rmse, rel, rmseV, relV = compute_metrics(Yp, Yt)

    metrics_rows.append(
        {
            "case": tag,
            "shape": c.paths.shape,
            "Re": float(c.paths.Re),
            "N_points": int(X.shape[0]),
            "RMSE_u": float(rmse[0]),
            "RMSE_v": float(rmse[1]),
            "RMSE_p": float(rmse[2]),
            "RMSE_Vmag": float(rmseV),
            "RelL2_u": float(rel[0]),
            "RelL2_v": float(rel[1]),
            "RelL2_p": float(rel[2]),
            "RelL2_Vmag": float(relV),
            "inference_s": infer_s,
        }
    )

    x = X[:, 0]
    y = X[:, 1]
    phi = X[:, 3]

    u_t, v_t, p_t = Yt[:, 0], Yt[:, 1], Yt[:, 2]
    u_p, v_p, p_p = Yp[:, 0], Yp[:, 1], Yp[:, 2]

    V_t = np.sqrt(u_t * u_t + v_t * v_t)
    V_p = np.sqrt(u_p * u_p + v_p * v_p)

    Xb = c.bc.get("body", {}).get("X", None)
    body_xy = None
    if Xb is not None and len(Xb) > 0:
        Xb = np.asarray(Xb, dtype=np.float64)
        body_xy = (Xb[:, 0], Xb[:, 1])

    out_3x3 = OUT_DIR / f"UVP_3x3_CFD_PINN_ERR_{tag}.png"
    save_3x3_fields(x, y, phi, u_t, v_t, p_t, u_p, v_p, p_p, tag, out_3x3, body_xy=body_xy, infer_s=infer_s)
    stamp_text_on_png(out_3x3, f"inference={infer_s:.3f}s")

    out_vmag = OUT_DIR / f"Vmag_1x3_CFD_PINN_ERR_{tag}.png"
    save_vmag_1x3(x, y, phi, V_t, V_p, tag, out_vmag, body_xy=body_xy, infer_s=infer_s)
    stamp_text_on_png(out_vmag, f"inference={infer_s:.3f}s")

    print(f"[OK] {tag} | inference={infer_s:.3f}s | saved figures")

print(f"[DONE] Field evaluation finished in {time.perf_counter() - t_eval0:.2f}s")

df_metrics = pd.DataFrame(metrics_rows).sort_values(["shape", "Re"]).set_index("case")
metrics_png = OUT_DIR / "FieldMetrics_ALL_CASES.png"
save_table_png(df_metrics.round(6), "Field Metrics (u,v,p,|V|) — All PASS Cases", metrics_png, font_size=10)
print("[SAVED]", metrics_png)


# ============================================================
# PART 2 — CASE-LEVEL QUANTITIES
# ============================================================
integral_rows = []

for c in cases:
    tag = f"{c.paths.shape}_Re{c.paths.Re}"
    cs = get_case_summary(c)

    fv = cs.get("final_values", {})
    if not isinstance(fv, dict):
        fv = {}

    cd_meta = pick_first_numeric(fv, ["Cd_final_used", "Cd_mean", "Cd_last", "Cd"])
    cl_meta = pick_first_numeric(fv, ["Cl_final_used", "Cl_mean", "Cl_last", "Cl"])
    dp_meta = pick_first_numeric(fv, ["deltap_nd_final_used", "deltap_nd_mean", "deltap_nd_last", "deltap_nd"])
    la_meta = pick_first_numeric(fv, ["La_nd_final_used", "La_nd_mean", "La_nd"])

    X = c.field_X.astype(np.float64)
    Yt = c.field_Y.astype(np.float64)

    with torch.no_grad():
        Yp = model(torch.tensor(X, dtype=DTYPE, device=DEVICE)).cpu().numpy()

    x = X[:, 0]
    y = X[:, 1]
    phi = X[:, 3]
    u_cfd = Yt[:, 0]
    u_pinn = Yp[:, 0]

    Xb = c.bc.get("body", {}).get("X", None)

    cd_cfd_field = float("nan")
    cl_cfd_field = float("nan")
    cd_pinn = float("nan")
    cl_pinn = float("nan")

    if Xb is not None and len(Xb) > 0:
        Xb = np.asarray(Xb, dtype=np.float64)
        cd_cfd_field, cl_cfd_field = cfd_cdcl_from_body_offset(
            c,
            Xb_np=Xb,
            Re=float(c.paths.Re),
            y_bot=y_bot,
            y_top=y_top,
        )
        cd_pinn, cl_pinn = pinn_cdcl_from_body_offset(
            model,
            c,
            Xb_np=Xb,
            Re=float(c.paths.Re),
            y_bot=y_bot,
            y_top=y_top,
        )

    front_nd, rear_nd = get_reference_probe_points_nd(cs)

    dp_cfd_field = float("nan")
    dp_pinn = float("nan")
    if front_nd is not None and rear_nd is not None:
        dp_cfd_field = deltap_from_probe_points_cfd_field(c, front_nd, rear_nd)
        dp_pinn = deltap_from_probe_points_pinn(model, c, front_nd, rear_nd, float(c.paths.Re))

    D_m = pick_first_numeric(cs.get("scales", {}), ["D_m"])
    if not np.isfinite(D_m) or D_m <= 0:
        D_m = 0.1

    y_center_nd = get_centerline_y_nd(cs, D_m=float(D_m), y_bot=y_bot, y_top=y_top)
    xe_nd = get_xe_nd(cs)

    la_cfd_field = La_from_centerline_field(x, y, phi, u_cfd, y_center_nd=y_center_nd, xe_nd=xe_nd, nx=1400)
    la_pinn = La_from_centerline_field(x, y, phi, u_pinn, y_center_nd=y_center_nd, xe_nd=xe_nd, nx=1400)

    df_case = pd.DataFrame(
        {
            "CFD meta": [cd_meta, cl_meta, dp_meta, la_meta],
            "CFD same method": [cd_cfd_field, cl_cfd_field, dp_cfd_field, la_cfd_field],
            "PINN same method": [cd_pinn, cl_pinn, dp_pinn, la_pinn],
        },
        index=["Cd_total", "Cl_total", "Δp_nd", "La_nd"],
    )

    out_png = OUT_DIR / f"IntegralTable_{tag}.png"
    save_table_png(
        df_case.round(6),
        title=f"Case Quantities | {tag}\n(CFD and PINN use same method for comparison)",
        out_path=out_png,
        font_size=12,
    )
    print("[SAVED]", out_png)

    integral_rows.append(
        {
            "case": tag,
            "Cd_CFD_meta": cd_meta,
            "Cd_CFD_same": cd_cfd_field,
            "Cd_PINN_same": cd_pinn,
            "Cl_CFD_meta": cl_meta,
            "Cl_CFD_same": cl_cfd_field,
            "Cl_PINN_same": cl_pinn,
            "Δp_nd_CFD_meta": dp_meta,
            "Δp_nd_CFD_same": dp_cfd_field,
            "Δp_nd_PINN_same": dp_pinn,
            "La_nd_CFD_meta": la_meta,
            "La_nd_CFD_same": la_cfd_field,
            "La_nd_PINN_same": la_pinn,
        }
    )

df_sum = pd.DataFrame(integral_rows).set_index("case")
out_sum = OUT_DIR / "IntegralTable_ALL_CASES.png"
save_table_png(
    df_sum.round(6),
    title="Case Quantities Comparison (All PASS Cases)\nMeta vs same-method CFD vs same-method PINN",
    out_path=out_sum,
    font_size=10,
)
print("[SAVED]", out_sum)

print("\n[DONE] Evaluation finished.")
print("PNG outputs saved in:", OUT_DIR)
print("Using checkpoint:", CKPT_PATH)

Using checkpoint: D:\Projects\Simulations\AI FInal\Code\Main Structure\checkpoints_stage5\stage5_pinn_cpu.pt True
OUT_DIR: D:\Projects\Simulations\AI FInal\Code\Results\Evaluation

=== AUDIT SUMMARY ===
root: D:\Projects\Simulations\AI FInal\Ansys\Shapes
PASS: 9 | FAIL: 0
report: D:\Projects\Simulations\AI FInal\Ansys\Shapes\_audit_report.csv

=== CASES USED FOR TRAINING (PASS) ===
- Circle | Re=10 | case_id=Circle_2D_Re10 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Circle | Re=20 | case_id=Circle_2D_Re20 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Circle | Re=30 | case_id=Circle_2D_Re30 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Hexagon | Re=10 | case_id=Hexagon_2D_Re10 | field_core_rows=184614 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,Del